# Module 1: Baseline Taxi Duration Prediction Model

This notebook downloads NYC TLC Green Taxi data, builds a baseline feature engineering pipeline, trains a Linear Regression model, outputs validation metrics to `reports/module-1.md`, and saves the trained model artifact to `models/baseline.pkl`.

In [ ]:
import os
import pickle
import urllib.request
import pandas as pd
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

# Ensure project directories exist
os.makedirs('../data', exist_ok=True)
os.makedirs('../reports', exist_ok=True)
os.makedirs('../models', exist_ok=True)

In [ ]:
# 1. Download 1 month of NYC TLC Green Taxi trip data (Jan 2023 Parquet)
data_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2023-01.parquet'
parquet_path = '../data/green_tripdata_2023-01.parquet'

if not os.path.exists(parquet_path):
    print('Downloading green taxi dataset...')
    urllib.request.urlretrieve(data_url, parquet_path)
    print('Download complete.')
else:
    print('File already exists locally.')

In [ ]:
# 2. Load data and calculate target variable
df = pd.read_parquet(parquet_path)

# Calculate duration in minutes
df['duration'] = (df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']).dt.total_seconds() / 60.0

# Filter outlier durations (keep trips between 1 and 60 minutes)
df = df[(df['duration'] >= 1) & (df['duration'] <= 60)].copy()

# 3. Engineer features: PU_DO (pickup-dropoff pair) and trip_distance
df['PU_DO'] = df['PULocationID'].astype(str) + '_' + df['DOLocationID'].astype(str)
features = ['PU_DO', 'trip_distance']

In [ ]:
# 4. Prepare Train / Validation split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

train_dicts = train_df[features].to_dict(orient='records')
val_dicts = val_df[features].to_dict(orient='records')

# Vectorize categorical and numerical features
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

y_train = train_df['duration'].values
y_val = val_df['duration'].values

In [ ]:
# 5. Train Baseline Model
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate on Validation set
y_pred = model.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae = mean_absolute_error(y_val, y_pred)

print(f'Validation RMSE: {rmse:.4f}')
print(f'Validation MAE:  {mae:.4f}')

In [ ]:
# 6. Write validation RMSE and MAE at the top of reports/module-1.md
report_path = '../reports/module-1.md'
metrics_header = f"# Module 1 Report\n\n## Baseline Model Metrics\n- **Validation RMSE:** {rmse:.4f}\n- **Validation MAE:** {mae:.4f}\n\n"

existing_content = ''
if os.path.exists(report_path):
    with open(report_path, 'r') as f:
        existing_content = f.read()

with open(report_path, 'w') as f:
    f.write(metrics_header + existing_content)

print(f'Successfully wrote metrics to {report_path}')

In [ ]:
# 7. Save fitted model artifact to models/baseline.pkl
model_path = '../models/baseline.pkl'
with open(model_path, 'wb') as f_out:
    pickle.dump((dv, model), f_out)

print(f'Fitted model saved to {model_path}')